# SR3: revisions and the policy curve

Training: May 2018–2022. Reused validation: 2023–2024. Final holdout: 2025 onwards, excluded. No downloads.

Run all cells. Tables, plotted data, figures and trial records are written automatically.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'pyproject.toml').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from tools.notebook_experiments import original_eda, curve_experiment
RUN_ID = 'sr3-curve-notebook-v1'
OUT = ROOT / 'results' / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .2})
print('Outputs:', OUT)

## Original hypothesis: distribution and IC

Histograms retain outliers. IC relates revision pressure to ten-session rate changes. Targets overlap; monthly IC requires five events. The parameter grid shows selection risk.

In [ ]:
display(original_eda(OUT))
display(pd.read_csv(ROOT / 'results/sr3-macro-revisions-v1/training-selected.csv'))

## Next hypothesis: curve repricing

Test twelve-month minus six-month forward-rate changes. Fixed decay: 30 days; clipping: 3; holding: 10 sessions. Ridge penalty is selected using 2021–2022 only. Positive forecasts sell the far contract and buy the near contract. Each leg pays costs. Full specification: `research/hypotheses/sr3-curve-revisions.md`.

In [ ]:
metrics, forecasts, selection = curve_experiment(OUT)
display(selection)
display(forecasts)
display(metrics[(metrics['sample'] == 'reused_validation') & (metrics.cost_bp_per_leg == 1)].sort_values(['alpha', 'threshold']))

## Cost sensitivity and annual contribution

In [ ]:
grid = metrics[metrics['sample'].eq('reused_validation') & metrics.threshold.eq(1)].pivot(index='alpha', columns='cost_bp_per_leg', values='net_pnl')
display(grid)
fig, ax = plt.subplots(figsize=(9, 4))
for alpha, row in grid.iterrows():
    ax.plot(grid.columns, row, marker='o', label=f'alpha={alpha}')
ax.axhline(0, color='black', lw=.8)
ax.set(xlabel='Round-trip cost per leg (bp)', ylabel='Validation net P&L ($)')
ax.legend(); fig.tight_layout(); fig.savefig(OUT / 'figures/cost-sensitivity.png', dpi=150); plt.show()
grid.to_csv(OUT / 'cost-sensitivity.csv')
chosen = int(selection.sort_values(['inner_MSE', 'alpha']).iloc[0].alpha)
r = pd.read_csv(OUT / f'returns-a{chosen}-t1-c1-reused_validation.csv')
r['year'] = pd.to_datetime(r.date).dt.year
yearly = r.groupby('year').agg(net_pnl=('pnl','sum'), sessions=('pnl','size'), mean_exposure=('gross_contract_exposure','mean'))
yearly.to_csv(OUT / 'annual-contributions.csv'); display(yearly)

## Decision

Compare forecast MSE with the constant baseline; inspect IC, coefficient sign, trade count, costs and neighbouring thresholds. A positive cell selected after viewing validation is exploratory. Further confirmatory testing requires a frozen specification and separate holdout access.

In [ ]:
print((OUT / 'REPORT.md').read_text())
print('Saved:', OUT)